# LeetCode 759 — Employee Free Time bằng Python

- Link: https://leetcode.com/problems/employee-free-time/
- Độ khó: **Hard**
- Chủ đề: Đoạn (Intervals), Sắp xếp, Gộp đoạn (Merge Intervals), Heap / trộn k danh sách, Sweep line (quét sự kiện)

> **Ghi chú (bài Premium):** đề bài dưới đây được viết lại theo hiểu biết về bài toán (định nghĩa lớp `Interval`, ví dụ, ràng buộc). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại phát biểu và chữ ký hàm chính thức.

Skeleton trên LeetCode (Python3):

```python
"""
# Definition for an Interval.
class Interval:
    def __init__(self, start: int = None, end: int = None):
        self.start = start
        self.end = end
"""

class Solution:
    def employeeFreeTime(self, schedule: '[[Interval]]') -> '[Interval]':
```

## 1. Giới thiệu & mục tiêu học

Mỗi nhân viên có một lịch làm việc gồm các khoảng thời gian bận. Ta cần tìm những khoảng thời gian (độ dài dương, hữu hạn) mà **tất cả mọi người đều rảnh**. Đây là bài "gộp đoạn" mặc áo mới: thời gian rảnh chung chính là **các khe hở** giữa các đoạn bận sau khi gộp lại.

Sau notebook này bạn sẽ:

1. Mô hình hoá "rảnh chung" = phần bù của **hợp** các đoạn bận.
2. Viết lời giải đánh dấu dòng thời gian (baseline) và hiểu vì sao nó không dùng được với toạ độ `10^8`.
3. Thành thạo **gộp đoạn sau khi sắp xếp** và tìm khe hở.
4. Tận dụng việc mỗi lịch **đã sắp xếp** để trộn bằng **heap** trong `O(N log K)`.
5. Dùng **sweep line** (đếm số người đang bận) — kỹ thuật dùng lại được cho rất nhiều bài lịch.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `schedule`, trong đó `schedule[i]` là danh sách các đoạn `Interval(start, end)` mà nhân viên `i` **bận**. Danh sách của mỗi người **đã sắp xếp** và các đoạn của cùng một người **không giao nhau**. Trả về danh sách các đoạn **rảnh chung** của mọi nhân viên, có **độ dài dương**, **hữu hạn**, và **sắp xếp tăng dần**.

Ta không xét các khoảng vô hạn như `[-inf, 1]` hay `[10, inf]`. Đoạn `[a, b]` hiểu là khoảng thời gian từ `a` đến `b`; hai đoạn `[1, 2]` và `[2, 3]` liền nhau không tạo ra khoảng rảnh (độ dài 0).

**Ví dụ 1**

```text
Input: schedule = [[[1,2],[5,6]],[[1,3]],[[4,10]]]
Output: [[3,4]]
Giải thích: có 3 nhân viên; khoảng rảnh chung hữu hạn duy nhất là [3,4].
```

**Ví dụ 2**

```text
Input: schedule = [[[1,3],[6,7]],[[2,4]],[[2,5],[9,12]]]
Output: [[5,6],[7,9]]
```

(Trên LeetCode, input/output là đối tượng `Interval`, chứ không phải list; `[1,2]` chỉ là cách viết gọn.)

**Ràng buộc**

- `1 <= schedule.length, schedule[i].length <= 50`
- `0 <= schedule[i][j].start < schedule[i][j].end <= 10^8`

Ta tự dựng lớp `Interval` và các hàm chuyển đổi để chạy được trong notebook.

In [1]:
from typing import List
import heapq

class Interval:
    def __init__(self, start: int = None, end: int = None):
        self.start = start
        self.end = end

    def __repr__(self):
        return f"[{self.start},{self.end}]"

def to_schedule(raw):
    """[[[1,2],[5,6]], ...] -> [[Interval, ...], ...]"""
    return [[Interval(s, e) for s, e in emp] for emp in raw]

def to_lists(intervals):
    return [[iv.start, iv.end] for iv in intervals]

EXAMPLES = [
    ([[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]], [[3, 4]]),
    ([[[1, 3], [6, 7]], [[2, 4]], [[2, 5], [9, 12]]], [[5, 6], [7, 9]]),
]

sch = to_schedule(EXAMPLES[0][0])
print(sch)

[[[1,2], [5,6]], [[1,3]], [[4,10]]]


## 3. Kiến thức nền

### 3.1. Lớp đơn giản và `__repr__`

Lớp `Interval` chỉ giữ hai thuộc tính. Định nghĩa `__repr__` giúp `print` hiển thị gọn `[1,2]` thay vì `<__main__.Interval object at ...>`.

In [2]:
iv = Interval(3, 4)
print(iv, iv.start, iv.end)
print([Interval(1, 2), Interval(5, 6)])

[3,4] 3 4
[[1,2], [5,6]]


### 3.2. Làm phẳng danh sách lồng và sắp theo khoá

`[iv for emp in schedule for iv in emp]` làm phẳng 2 tầng. `sort(key=lambda iv: iv.start)` sắp theo thời điểm bắt đầu.

In [3]:
flat = [iv for emp in to_schedule(EXAMPLES[1][0]) for iv in emp]
print(flat)
flat.sort(key=lambda iv: iv.start)
print(flat)

[[1,3], [6,7], [2,4], [2,5], [9,12]]
[[1,3], [2,4], [2,5], [6,7], [9,12]]


### 3.3. Gộp đoạn (Merge Intervals — LeetCode 56)

Sắp theo `start`. Duyệt: nếu đoạn mới bắt đầu **trước hoặc đúng** lúc đoạn đang gộp kết thúc (`s <= cur_end`) thì kéo dài `cur_end = max(cur_end, e)`; ngược lại, đóng đoạn cũ, mở đoạn mới.

In [4]:
def merge(pairs):
    pairs = sorted(pairs)
    out = []
    for s, e in pairs:
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out

print(merge([[1, 3], [2, 6], [8, 10], [10, 12], [15, 18]]))

[[1, 6], [8, 12], [15, 18]]


### 3.4. `heapq`: hàng đợi ưu tiên (min-heap)

`heapq.heappush(h, x)` và `heapq.heappop(h)` trong `O(log size)`; phần tử nhỏ nhất luôn ở `h[0]`. Đẩy **tuple** `(start, emp_id, idx)` để heap so sánh theo `start` trước; `emp_id`, `idx` giúp tránh so sánh trực tiếp hai đối tượng `Interval` (sẽ lỗi `TypeError`).

In [5]:
h = []
for item in [(5, 0, 1), (1, 1, 0), (2, 2, 0)]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(3)])

try:
    heapq.heappush(h, (1, Interval(1, 2)))
    heapq.heappush(h, (1, Interval(1, 3)))    # start bằng nhau -> so sánh Interval
except TypeError as ex:
    print("TypeError:", ex)

[(1, 1, 0), (2, 2, 0), (5, 0, 1)]
TypeError: '<' not supported between instances of 'Interval' and 'Interval'


### 3.5. Sweep line: sự kiện +1 / -1

Biến mỗi đoạn `[s, e]` thành hai sự kiện: `(s, +1)` (một người bắt đầu bận) và `(e, -1)` (kết thúc). Sắp các sự kiện theo thời gian, duyệt và cộng dồn → biết **số người đang bận** tại mỗi thời điểm. Chỗ nào bộ đếm bằng 0 (giữa hai thời điểm khác nhau) là rảnh chung.

In [6]:
events = []
for s, e in [[1, 3], [2, 4], [6, 7]]:
    events += [(s, +1), (e, -1)]
events.sort()
busy = 0
for t, d in events:
    busy += d
    print(f"t={t}: {'+1' if d > 0 else '-1'} -> đang bận {busy} người")

t=1: +1 -> đang bận 1 người
t=2: +1 -> đang bận 2 người
t=3: -1 -> đang bận 1 người
t=4: -1 -> đang bận 0 người
t=6: +1 -> đang bận 1 người
t=7: -1 -> đang bận 0 người


## 4. Ý tưởng nền tảng / trực giác

**Quan sát:** "rảnh chung của mọi người" = "không ai bận" = nằm ngoài **hợp** của tất cả các đoạn bận. Ta **không cần biết đoạn nào của ai** — chỉ cần gộp tất cả lại.

### Tính tay ví dụ 2

Các đoạn bận (gộp mọi người): `[1,3], [6,7], [2,4], [2,5], [9,12]`.

Sắp theo start: `[1,3], [2,4], [2,5], [6,7], [9,12]`.

Gộp:

- `[1,3]` → đang gộp `[1,3]`.
- `[2,4]`: `2 <= 3` → `[1,4]`.
- `[2,5]`: `2 <= 4` → `[1,5]`.
- `[6,7]`: `6 > 5` → đóng `[1,5]`; **khe hở `[5,6]`**; mở `[6,7]`.
- `[9,12]`: `9 > 7` → đóng `[6,7]`; **khe hở `[7,9]`**; mở `[9,12]`.

Kết quả: `[[5,6],[7,9]]`. ✔

Hình dung trên trục thời gian:

```text
thời gian: 1  2  3  4  5  6  7  8  9 10 11 12
NV 0:      [=====]           [==]
NV 1:         [=====]
NV 2:         [========]              [========]
hợp:       [===========]     [==]     [========]
rảnh:                  [==]     [=====]
```

In [7]:
busy_all = [[1, 3], [6, 7], [2, 4], [2, 5], [9, 12]]
merged = merge(busy_all)
gaps = [[merged[i][1], merged[i + 1][0]] for i in range(len(merged) - 1)]
print("hợp:", merged, "-> rảnh:", gaps)

hợp: [[1, 5], [6, 7], [9, 12]] -> rảnh: [[5, 6], [7, 9]]


## 5. Cách 1 — Brute Force: đánh dấu từng đơn vị thời gian

**Ý tưởng.** Chia trục thời gian thành các ô `[t, t+1)` với `t` từ `min start` tới `max end - 1`. Ô `t` bận nếu có đoạn với `start <= t < end`. Gom các ô rảnh liên tiếp thành đoạn. Chỉ dùng được khi toạ độ nhỏ.

In [8]:
def brute(raw):
    pairs = [p for emp in raw for p in emp]
    lo = min(s for s, _ in pairs)
    hi = max(e for _, e in pairs)
    busy = [False] * (hi - lo)
    for s, e in pairs:
        for t in range(s, e):
            busy[t - lo] = True
    res, t = [], 0
    while t < hi - lo:
        if busy[t]:
            t += 1
            continue
        u = t
        while u < hi - lo and not busy[u]:
            u += 1
        res.append([t + lo, u + lo])     # các ô [t, u) đều rảnh
        t = u
    return res

for raw, exp in EXAMPLES:
    print(brute(raw), exp)

[[3, 4]] [[3, 4]]
[[5, 6], [7, 9]] [[5, 6], [7, 9]]


**Độ phức tạp.** `O(N · W)` với `W` = độ rộng toạ độ (đến `10^8`) và `N` = tổng số đoạn. Bộ nhớ `O(W)`. Với `10^8` → quá chậm và quá tốn bộ nhớ.

Lưu ý: vì đoạn bên trong lịch không tạo khoảng rảnh ở ngoài `[lo, hi]`, các ô ngoài rìa luôn bận ở hai đầu → kết quả chỉ gồm khoảng hữu hạn.

**Nút thắt.** Ta làm việc trên **từng đơn vị thời gian**, trong khi thông tin chỉ thay đổi tại **đầu mút** các đoạn (≤ 2·2500 = 5000 điểm). Cần thuật toán phụ thuộc **số đoạn**, không phụ thuộc độ lớn toạ độ.

## 6. Bước đột phá

**Insight.** Chỉ các đầu mút mới quan trọng. Sắp tất cả đoạn bận theo `start` rồi quét với biến `prev_end` = thời điểm muộn nhất mà ai đó còn bận tính đến giờ:

- Nếu đoạn tiếp theo bắt đầu **sau** `prev_end` (`s > prev_end`) → khoảng `[prev_end, s]` không ai bận → **rảnh chung**.
- Luôn cập nhật `prev_end = max(prev_end, e)`.

**Bất biến.** Khi xét đoạn thứ `i` (theo thứ tự start), `prev_end` = `max(end)` của các đoạn `0..i-1`, và mọi thời điểm trong `[start_0, prev_end]` mà không thuộc khe hở đã ghi đều có người bận.

**Insight thứ hai (tận dụng dữ liệu đã sắp).** Mỗi nhân viên đã có danh sách sắp sẵn → không cần sắp lại tất cả; chỉ cần **trộn `K` danh sách đã sắp** bằng heap, giống *23. Merge k Sorted Lists*.

**Quy tắc nếu… thì…**

- Nếu `s > prev_end` → thêm `Interval(prev_end, s)`.
- Nếu `s <= prev_end` (kể cả `s == prev_end`, chạm nhau) → không có khe hở.
- Luôn `prev_end = max(prev_end, e)` — **không** gán thẳng `prev_end = e` (đoạn sau có thể nằm lọt trong đoạn trước).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Làm phẳng + sắp xếp + quét khe hở

**Giải thích.** Gom mọi đoạn vào một list, sắp theo `start`, quét như mục 6.

In [9]:
def free_sort(raw):
    pairs = sorted(p for emp in raw for p in emp)
    res = []
    prev_end = pairs[0][1]
    for s, e in pairs[1:]:
        if s > prev_end:
            res.append([prev_end, s])
        prev_end = max(prev_end, e)
    return res

for raw, exp in EXAMPLES:
    print(free_sort(raw), exp)

[[3, 4]] [[3, 4]]
[[5, 6], [7, 9]] [[5, 6], [7, 9]]


**Code nộp LeetCode (Cách 2)** — làm việc trực tiếp với đối tượng `Interval`.

In [10]:
class SolutionSort:
    def employeeFreeTime(self, schedule: '[[Interval]]') -> '[Interval]':
        ivs = sorted((iv for emp in schedule for iv in emp), key=lambda iv: iv.start)
        res = []
        prev_end = ivs[0].end
        for iv in ivs[1:]:
            if iv.start > prev_end:
                res.append(Interval(prev_end, iv.start))
            prev_end = max(prev_end, iv.end)
        return res

for raw, exp in EXAMPLES:
    print(to_lists(SolutionSort().employeeFreeTime(to_schedule(raw))), exp)

[[3, 4]] [[3, 4]]
[[5, 6], [7, 9]] [[5, 6], [7, 9]]


**Độ phức tạp.** `N` = tổng số đoạn (≤ 2500). Sắp xếp `O(N log N)`, quét `O(N)`. Bộ nhớ `O(N)`.

### 7.2. Cách 3 — Trộn K danh sách đã sắp bằng heap

**Giải thích.** Đẩy đoạn **đầu tiên** của mỗi nhân viên vào heap với khoá `(start, emp, idx)`. Mỗi lần pop đoạn có `start` nhỏ nhất, xử lý như Cách 2, rồi đẩy đoạn **kế tiếp** của cùng nhân viên (nếu có). Heap chỉ chứa tối đa `K` phần tử.

In [11]:
def free_heap(raw):
    h = [(emp[0][0], i, 0) for i, emp in enumerate(raw) if emp]
    heapq.heapify(h)
    res = []
    prev_end = None
    while h:
        s, i, j = heapq.heappop(h)
        e = raw[i][j][1]
        if prev_end is not None and s > prev_end:
            res.append([prev_end, s])
        prev_end = e if prev_end is None else max(prev_end, e)
        if j + 1 < len(raw[i]):
            heapq.heappush(h, (raw[i][j + 1][0], i, j + 1))
    return res

for raw, exp in EXAMPLES:
    print(free_heap(raw), exp)

[[3, 4]] [[3, 4]]
[[5, 6], [7, 9]] [[5, 6], [7, 9]]


**Code nộp LeetCode (Cách 3).**

In [12]:
class SolutionHeap:
    def employeeFreeTime(self, schedule: '[[Interval]]') -> '[Interval]':
        import heapq
        h = [(emp[0].start, i, 0) for i, emp in enumerate(schedule) if emp]
        heapq.heapify(h)
        res = []
        prev_end = None
        while h:
            s, i, j = heapq.heappop(h)
            iv = schedule[i][j]
            if prev_end is not None and s > prev_end:
                res.append(Interval(prev_end, s))
            prev_end = iv.end if prev_end is None else max(prev_end, iv.end)
            if j + 1 < len(schedule[i]):
                heapq.heappush(h, (schedule[i][j + 1].start, i, j + 1))
        return res

for raw, exp in EXAMPLES:
    print(to_lists(SolutionHeap().employeeFreeTime(to_schedule(raw))), exp)

[[3, 4]] [[3, 4]]
[[5, 6], [7, 9]] [[5, 6], [7, 9]]


**Độ phức tạp.** Mỗi đoạn vào/ra heap một lần, heap cỡ ≤ `K` → `O(N log K)`. Bộ nhớ `O(K)` cho heap (+ kết quả). Với `K` nhỏ so với `N` thì nhanh hơn Cách 2 về lý thuyết.

### 7.3. Cách 4 — Sweep line đếm số người bận

**Giải thích.** Tạo sự kiện `(s, +1)`, `(e, -1)`. Sắp theo thời gian. **Gom các sự kiện cùng thời điểm** rồi mới xét bộ đếm. Khi bộ đếm **về 0** tại thời điểm `t`, đánh dấu `free_start = t`; khi bộ đếm từ 0 **tăng lên** tại thời điểm `t' > free_start`, ghi khoảng rảnh `[free_start, t']`.

Gom theo thời điểm giúp xử lý trường hợp chạm nhau `[1,2]` và `[2,3]`: tại `t = 2` có `-1` và `+1`, sau khi gom bộ đếm vẫn là 1 → không có khoảng rảnh.

Một cách khác là mã hoá sự kiện sao cho tại cùng thời điểm, `+1` được xử lý **trước** `-1` (ví dụ dùng tuple `(t, 0)` cho bắt đầu và `(t, 1)` cho kết thúc); khi đó bộ đếm không bao giờ chạm 0 tại chỗ hai đoạn chạm nhau. Nhưng gom nhóm theo thời điểm là cách rõ ràng nhất và ít dễ sai nhất.

In [13]:
from itertools import groupby

def free_sweep(raw):
    events = []
    for emp in raw:
        for s, e in emp:
            events.append((s, 1))
            events.append((e, -1))
    events.sort()
    res = []
    busy = 0
    free_start = None
    for t, grp in groupby(events, key=lambda x: x[0]):
        before = busy
        busy += sum(d for _, d in grp)
        if before == 0 and busy > 0 and free_start is not None and t > free_start:
            res.append([free_start, t])
        if before > 0 and busy == 0:
            free_start = t
    return res

for raw, exp in EXAMPLES:
    print(free_sweep(raw), exp)

[[3, 4]] [[3, 4]]
[[5, 6], [7, 9]] [[5, 6], [7, 9]]


**Code nộp LeetCode (Cách 4).**

In [14]:
class SolutionSweep:
    def employeeFreeTime(self, schedule: '[[Interval]]') -> '[Interval]':
        delta = {}
        for emp in schedule:
            for iv in emp:
                delta[iv.start] = delta.get(iv.start, 0) + 1
                delta[iv.end] = delta.get(iv.end, 0) - 1
        res = []
        busy = 0
        free_start = None
        for t in sorted(delta):
            before = busy
            busy += delta[t]
            if before == 0 and busy > 0 and free_start is not None:
                res.append(Interval(free_start, t))
            if before > 0 and busy == 0:
                free_start = t
        return res

for raw, exp in EXAMPLES:
    print(to_lists(SolutionSweep().employeeFreeTime(to_schedule(raw))), exp)

[[3, 4]] [[3, 4]]
[[5, 6], [7, 9]] [[5, 6], [7, 9]]


Ở bản nộp, ta gom bằng `dict` `delta[t]` (tổng thay đổi tại `t`) — vừa gom nhóm vừa đếm. Vì các khoá `t` khác nhau và tăng dần, `t > free_start` luôn đúng nên không cần kiểm tra.

**Độ phức tạp.** `O(N log N)` cho sắp `2N` sự kiện. Bộ nhớ `O(N)`.

## 8. TRACE từng bước

Trace **Cách 2** trên ví dụ 2. Đoạn đã sắp: `[1,3], [2,4], [2,5], [6,7], [9,12]`.

| bước | đoạn | prev_end trước | s > prev_end? | khe hở thêm | prev_end sau |
|---|---|---|---|---|---|
| khởi tạo | [1,3] | — | — | — | 3 |
| 1 | [2,4] | 3 | 2 > 3? không | — | 4 |
| 2 | [2,5] | 4 | không | — | 5 |
| 3 | [6,7] | 5 | 6 > 5 có | [5,6] | 7 |
| 4 | [9,12] | 7 | 9 > 7 có | [7,9] | 12 |

In [15]:
def trace_sort(raw):
    pairs = sorted(p for emp in raw for p in emp)
    print("đã sắp:", pairs)
    res = []
    prev_end = pairs[0][1]
    print(f"khởi tạo với {pairs[0]}: prev_end = {prev_end}")
    for step, (s, e) in enumerate(pairs[1:], 1):
        gap = s > prev_end
        if gap:
            res.append([prev_end, s])
        old = prev_end
        prev_end = max(prev_end, e)
        print(f"b{step}: đoạn [{s},{e}]  prev_end {old:>2} -> {prev_end:>2}  "
              f"{'KHE HỞ [' + str(old) + ',' + str(s) + ']' if gap else 'không khe hở'}  res={res}")
    return res

trace_sort(EXAMPLES[1][0])

đã sắp: [[1, 3], [2, 4], [2, 5], [6, 7], [9, 12]]
khởi tạo với [1, 3]: prev_end = 3
b1: đoạn [2,4]  prev_end  3 ->  4  không khe hở  res=[]
b2: đoạn [2,5]  prev_end  4 ->  5  không khe hở  res=[]
b3: đoạn [6,7]  prev_end  5 ->  7  KHE HỞ [5,6]  res=[[5, 6]]
b4: đoạn [9,12]  prev_end  7 -> 12  KHE HỞ [7,9]  res=[[5, 6], [7, 9]]


[[5, 6], [7, 9]]

Trace **Cách 3 (heap)** trên ví dụ 1: quan sát nội dung heap (chỉ ≤ K = 3 phần tử).

In [16]:
def trace_heap(raw):
    h = [(emp[0][0], i, 0) for i, emp in enumerate(raw) if emp]
    heapq.heapify(h)
    res, prev_end = [], None
    while h:
        print("heap:", sorted(h), end="  ")
        s, i, j = heapq.heappop(h)
        e = raw[i][j][1]
        if prev_end is not None and s > prev_end:
            res.append([prev_end, s])
        prev_end = e if prev_end is None else max(prev_end, e)
        print(f"pop NV{i} đoạn [{s},{e}] -> prev_end={prev_end}, res={res}")
        if j + 1 < len(raw[i]):
            heapq.heappush(h, (raw[i][j + 1][0], i, j + 1))
    return res

trace_heap(EXAMPLES[0][0])

heap: [(1, 0, 0), (1, 1, 0), (4, 2, 0)]  pop NV0 đoạn [1,2] -> prev_end=2, res=[]
heap: [(1, 1, 0), (4, 2, 0), (5, 0, 1)]  pop NV1 đoạn [1,3] -> prev_end=3, res=[]
heap: [(4, 2, 0), (5, 0, 1)]  pop NV2 đoạn [4,10] -> prev_end=10, res=[[3, 4]]
heap: [(5, 0, 1)]  pop NV0 đoạn [5,6] -> prev_end=10, res=[[3, 4]]


[[3, 4]]

Trace **Cách 4 (sweep)** trên một ví dụ có đoạn chạm nhau: `[[[1,2]],[[2,3]],[[5,6]]]` → chỉ có `[3,5]`.

In [17]:
def trace_sweep(raw):
    delta = {}
    for emp in raw:
        for s, e in emp:
            delta[s] = delta.get(s, 0) + 1
            delta[e] = delta.get(e, 0) - 1
    busy, free_start, res = 0, None, []
    for t in sorted(delta):
        before = busy
        busy += delta[t]
        note = ""
        if before == 0 and busy > 0 and free_start is not None:
            res.append([free_start, t]); note = f"ghi rảnh [{free_start},{t}]"
        if before > 0 and busy == 0:
            free_start = t; note = f"mọi người rảnh từ {t}"
        print(f"t={t:>2} delta={delta[t]:+d} bận {before}->{busy} {note}")
    print("kết quả:", res)
    return res

trace_sweep([[[1, 2]], [[2, 3]], [[5, 6]]])

t= 1 delta=+1 bận 0->1 
t= 2 delta=+0 bận 1->1 
t= 3 delta=-1 bận 1->0 mọi người rảnh từ 3
t= 5 delta=+1 bận 0->1 ghi rảnh [3,5]
t= 6 delta=-1 bận 1->0 mọi người rảnh từ 6
kết quả: [[3, 5]]


[[3, 5]]

## 9. Code nộp LeetCode

Bản chọn: **Cách 2** (ngắn, rõ, `O(N log N)` với `N ≤ 2500`). Lớp `Interval` do LeetCode cung cấp sẵn — **không** khai báo lại khi nộp.

In [18]:
class Solution:
    def employeeFreeTime(self, schedule: '[[Interval]]') -> '[Interval]':
        # 1) gom mọi đoạn bận và sắp theo start
        ivs = sorted((iv for emp in schedule for iv in emp), key=lambda iv: iv.start)
        res = []
        prev_end = ivs[0].end          # thời điểm muộn nhất có người bận
        for iv in ivs[1:]:
            if iv.start > prev_end:    # khe hở: không ai bận trong (prev_end, start)
                res.append(Interval(prev_end, iv.start))
            prev_end = max(prev_end, iv.end)
        return res

for raw, exp in EXAMPLES:
    assert to_lists(Solution().employeeFreeTime(to_schedule(raw))) == exp
print("OK")

OK


Giải thích từng dòng:

- Generator `(iv for emp in schedule for iv in emp)` làm phẳng không tạo list trung gian; `sorted` tạo list mới.
- `prev_end = ivs[0].end`: đoạn đầu tiên mở đầu vùng bận. Không có khoảng rảnh nào trước nó (vì phải hữu hạn).
- `iv.start > prev_end`: dấu **lớn hơn hẳn** để loại khoảng độ dài 0.
- `max(prev_end, iv.end)`: xử lý đoạn nằm lọt bên trong.
- Không cần thêm khoảng rảnh sau đoạn cuối (vô hạn).

## 10. Vì sao thuật toán đúng

Gọi `U` là hợp của mọi đoạn bận. Một thời điểm là rảnh chung ⇔ nó **không thuộc** `U`. Phần bù hữu hạn của `U` trong `[min start, max end]` chính là các khe hở giữa các thành phần liên thông của `U`.

**Bất biến (Cách 2).** Sau khi xử lý các đoạn `0..i` (theo `start` tăng), đoạn `[start_0, prev_end]` bằng hợp của chúng **trừ đi** các khe hở đã ghi, và các khe hở đã ghi là chính xác.

- Khởi tạo: đúng với đoạn 0.
- Bước `i+1`: mọi đoạn chưa xét có `start >= start_{i+1}`. Nếu `start_{i+1} > prev_end`, không đoạn nào (đã xét hay chưa) phủ `(prev_end, start_{i+1})` → đó là một khe hở thật sự và **trọn vẹn** (không kéo dài hơn được vì `start_{i+1}` bị bận). Nếu `start_{i+1} <= prev_end`, đoạn mới nối liền vùng bận, không tạo khe hở. Cập nhật `prev_end` đúng bằng max.

Vì các khe hở được ghi theo thứ tự thời gian tăng dần, kết quả đã sắp. Cách 3 xử lý **cùng một thứ tự** (theo start) nên đúng theo cùng lập luận. Cách 4: bộ đếm bằng số đoạn chứa thời điểm ngay sau `t`; khoảng rảnh là nơi bộ đếm = 0 giữa hai thời điểm sự kiện.

## 11. Phân tích Time/Space

Gọi `N` = tổng số đoạn, `K` = số nhân viên, `W` = độ rộng toạ độ.

| Cách | Time | Space |
|---|---|---|
| 1. Đánh dấu ô | `O(N · W)` | `O(W)` |
| 2. Sắp + quét | `O(N log N)` | `O(N)` |
| 3. Heap trộn K danh sách | `O(N log K)` | `O(K)` (+ output) |
| 4. Sweep line | `O(N log N)` | `O(N)` |

In [19]:
import random, time
random.seed(759)

def gen_raw(K, per, max_t, max_len):
    raw = []
    for _ in range(K):
        t = random.randint(0, max_len)
        emp = []
        for _ in range(per):
            s = t + random.randint(0, max_len)
            e = s + random.randint(1, max_len)
            if e > max_t:
                break
            emp.append([s, e])
            t = e
        if not emp:
            emp = [[0, 1]]
        raw.append(emp)
    return raw

big = gen_raw(50, 50, 10 ** 8, 10 ** 5)
for name, f in [("sort", free_sort), ("heap", free_heap), ("sweep", free_sweep)]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{name:6} {len(r):5} khoảng rảnh  {1000 * (time.perf_counter() - t0):6.2f} ms")

sort       3 khoảng rảnh    1.11 ms
heap       3 khoảng rảnh    1.81 ms
sweep      3 khoảng rảnh    5.29 ms


## 12. Bộ test

In [20]:
methods = {
    "brute": brute,
    "sort": free_sort,
    "heap": free_heap,
    "sweep": free_sweep,
    "SolutionSort": lambda raw: to_lists(SolutionSort().employeeFreeTime(to_schedule(raw))),
    "SolutionHeap": lambda raw: to_lists(SolutionHeap().employeeFreeTime(to_schedule(raw))),
    "SolutionSweep": lambda raw: to_lists(SolutionSweep().employeeFreeTime(to_schedule(raw))),
    "Solution": lambda raw: to_lists(Solution().employeeFreeTime(to_schedule(raw))),
}

# 12.1 Ví dụ đề
for name, f in methods.items():
    for raw, exp in EXAMPLES:
        assert f(raw) == exp, (name, raw)

# 12.2 Edge cases
edge = [
    ([[[1, 2]]], []),                                  # 1 người, 1 đoạn
    ([[[1, 2], [3, 4]]], [[2, 3]]),                    # 1 người, có khe
    ([[[1, 2]], [[2, 3]]], []),                        # chạm nhau -> độ dài 0
    ([[[1, 10]], [[2, 3], [5, 6]]], []),               # lọt trong
    ([[[1, 3]], [[1, 3]]], []),                        # trùng nhau
    ([[[0, 1]], [[100, 101]]], [[1, 100]]),
    ([[[1, 2], [8, 9]], [[3, 4]], [[5, 6]]], [[2, 3], [4, 5], [6, 8]]),
]
for name, f in methods.items():
    for raw, exp in edge:
        assert f(raw) == exp, (name, raw, f(raw))
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [21]:
# 12.3 Ngẫu nhiên, đối chiếu brute force (toạ độ nhỏ)
random.seed(42)
for _ in range(500):
    raw = gen_raw(random.randint(1, 5), random.randint(1, 4), 40, 4)
    b = brute(raw)
    for name, f in methods.items():
        assert f(raw) == b, (name, raw, f(raw), b)
print("Đối chiếu ngẫu nhiên 500 test: OK")

Đối chiếu ngẫu nhiên 500 test: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Gán `prev_end = e` thay vì `max`

**Code sai:**

In [22]:
def wrong_no_max(raw):
    pairs = sorted(p for emp in raw for p in emp)
    res, prev_end = [], pairs[0][1]
    for s, e in pairs[1:]:
        if s > prev_end:
            res.append([prev_end, s])
        prev_end = e                    # SAI
    return res

raw = [[[1, 10]], [[2, 3], [5, 6]]]
print("sai:", wrong_no_max(raw), " đúng:", free_sort(raw))

sai: [[3, 5]]  đúng: []


**Vì sao sai:** đoạn `[2,3]` nằm **lọt** trong `[1,10]`; gán `prev_end = 3` làm ta "quên" rằng người kia bận tới 10, rồi báo nhầm `[3,5]` rảnh.

**Code đúng:** `prev_end = max(prev_end, e)`.

### Lỗi 2 — Dùng `>=` và sinh khoảng rảnh độ dài 0

**Code sai:**

In [23]:
def wrong_ge(raw):
    pairs = sorted(p for emp in raw for p in emp)
    res, prev_end = [], pairs[0][1]
    for s, e in pairs[1:]:
        if s >= prev_end:               # SAI
            res.append([prev_end, s])
        prev_end = max(prev_end, e)
    return res

raw = [[[1, 2]], [[2, 3]]]
print("sai:", wrong_ge(raw), " đúng:", free_sort(raw))

sai: [[2, 2]]  đúng: []


**Vì sao sai:** `[2,2]` có độ dài 0, đề yêu cầu khoảng rảnh **độ dài dương**.

**Code đúng:** `s > prev_end`.

### Lỗi 3 — Chỉ tìm khe hở **trong từng người** rồi giao lại sai cách / chỉ xét người đầu

**Code sai:**

In [24]:
def wrong_per_employee(raw):
    # SAI: lấy khe hở của từng người rồi hợp lại (union) thay vì giao (intersection)
    res = set()
    for emp in raw:
        for i in range(len(emp) - 1):
            if emp[i + 1][0] > emp[i][1]:
                res.add((emp[i][1], emp[i + 1][0]))
    return sorted(map(list, res))

raw = EXAMPLES[1][0]
print("sai:", wrong_per_employee(raw), " đúng:", free_sort(raw))

sai: [[3, 6], [5, 9]]  đúng: [[5, 6], [7, 9]]


**Vì sao sai:** rảnh chung phải là thời gian **mọi người** đều rảnh (giao), và còn phải tính cả người chỉ có một đoạn. Hợp các khe hở riêng cho ra cả những lúc người khác đang bận.

**Code đúng:** gộp **tất cả** đoạn bận rồi lấy khe hở của hợp (Cách 2).

### Lỗi 4 — Đẩy đối tượng `Interval` trực tiếp vào heap

**Code sai:**

In [25]:
def wrong_heap_objects(schedule):
    h = []
    for emp in schedule:
        for iv in emp:
            heapq.heappush(h, (iv.start, iv))     # SAI khi start trùng
    return [heapq.heappop(h)[1] for _ in range(len(h))]

try:
    wrong_heap_objects(to_schedule([[[1, 2]], [[1, 3]]]))
except TypeError as ex:
    print("TypeError:", ex)

TypeError: '<' not supported between instances of 'Interval' and 'Interval'


**Vì sao sai:** khi `start` bằng nhau, Python so sánh phần tử thứ hai của tuple — hai `Interval` không định nghĩa `<` → `TypeError`.

**Code đúng:** đẩy `(start, emp_id, idx)` (các số nguyên) như Cách 3.

## 14. Mẹo & nhận dạng pattern

- **"Thời gian rảnh chung / không ai bận"** → phần bù của **hợp** các đoạn → gộp đoạn.
- **"Mọi người đều bận"** (giao của các lịch) → sweep line với `busy == K`, hoặc hai con trỏ (*986. Interval List Intersections*).
- **Nhiều danh sách đã sắp** → heap trộn K danh sách, `O(N log K)`.
- **Đếm số đoạn phủ một điểm** → sweep line `+1/-1` (*253. Meeting Rooms II*, *732. My Calendar III*).
- **Chạm nhau có tính là giao không?** — luôn đọc kỹ: ở đây `[1,2]` và `[2,3]` **không** tạo khoảng rảnh.

## 15. Biến thể

1. **Có kèm giới hạn ngày làm việc** `[day_start, day_end]` → thêm khoảng rảnh ở hai rìa, cắt vào trong ngày.
2. **Tìm khoảng rảnh chung dài ít nhất `d`** (*1229. Meeting Scheduler*) → lọc kết quả theo độ dài.
3. **Ít nhất `m` trong `K` người rảnh** → sweep line, rảnh khi `busy <= K - m` (cần đếm theo **người**, mỗi người không có đoạn chồng nhau nên đếm đoạn = đếm người).
4. **Thời gian mọi người đều bận** → sweep line với `busy == K`.

In [26]:
# Biến thể 1: giới hạn trong [day_start, day_end]
def free_in_day(raw, day_start, day_end):
    pairs = sorted(p for emp in raw for p in emp)
    res, prev = [], day_start
    for s, e in pairs:
        s, e = max(s, day_start), min(e, day_end)
        if s >= e:
            continue
        if s > prev:
            res.append([prev, s])
        prev = max(prev, e)
    if prev < day_end:
        res.append([prev, day_end])
    return res

print(free_in_day(EXAMPLES[0][0], 0, 12))

# Biến thể 3: ít nhất m người rảnh
def free_at_least(raw, m):
    K = len(raw)
    delta = {}
    for emp in raw:
        for s, e in emp:
            delta[s] = delta.get(s, 0) + 1
            delta[e] = delta.get(e, 0) - 1
    ts = sorted(delta)
    res, busy = [], 0
    for a, b in zip(ts, ts[1:]):
        busy += delta[a]
        if K - busy >= m:
            if res and res[-1][1] == a:
                res[-1][1] = b
            else:
                res.append([a, b])
    return res

print(free_at_least(EXAMPLES[1][0], 2))
assert free_at_least(EXAMPLES[1][0], 3) == free_sort(EXAMPLES[1][0])

[[0, 1], [3, 4], [10, 12]]
[[1, 2], [4, 12]]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đánh dấu ô | Đánh dấu từng đơn vị thời gian | `O(N·W)` | `O(W)` | Toạ độ rất nhỏ, kiểm thử |
| 2. Sắp + quét | Gộp mọi đoạn theo start, ghi khe hở | `O(N log N)` | `O(N)` | Mặc định, dễ viết nhất |
| 3. Heap | Trộn K danh sách đã sắp | `O(N log K)` | `O(K)` | Khi K nhỏ, dữ liệu stream |
| 4. Sweep line | Đếm số người bận theo sự kiện | `O(N log N)` | `O(N)` | Khi cần tổng quát "≥ m người" |

## 17. Bài tập

**Bài 1 (đoán output).** `schedule = [[[1,5]],[[6,8],[10,11]],[[2,3],[12,15]]]` → ?

**Bài 2 (tính tay).** Với `[[[0,4],[7,9]],[[3,5]],[[10,12]]]`, viết bảng `prev_end` qua từng bước của Cách 2.

**Bài 3 (điền TODO).** Hoàn thiện `merge_intervals(pairs)` (LeetCode 56).

**Bài 4 (tự viết lại).** Viết lại Cách 3 (heap) nhưng **không** dùng chỉ số `j`, mà dùng `iter()` cho mỗi nhân viên.

**Bài 5 (mini challenge).** Viết `common_busy(raw)` trả về các khoảng mà **mọi** nhân viên đều bận (độ dài dương), đối chiếu brute force.

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def merge_intervals(pairs):
    # TODO: sắp theo start; nối nếu s <= out[-1][1]
    pass

# Bài 4 — TODO
def free_heap_iter(raw):
    # TODO: its = [iter(emp) for emp in raw]; heap chứa (s, e, i); pop rồi next(its[i], None)
    pass

# Bài 5 — TODO
def common_busy(raw):
    # TODO: sweep line, ghi khoảng khi busy == len(raw)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: gộp tất cả: `[1,5],[2,3],[6,8],[10,11],[12,15]`.
- Mức 2: hợp: `[1,5],[6,8],[10,11],[12,15]`.
- Mức 3: khe hở: `[[5,6],[8,10],[11,12]]`.

**Bài 2.**
- Mức 1: đã sắp: `[0,4],[3,5],[7,9],[10,12]`.
- Mức 2: `prev_end`: 4 → 5 → (khe [5,7]) 9 → (khe [9,10]) 12.
- Mức 3: kết quả `[[5,7],[9,10]]`.

**Bài 3.**
- Mức 1: `sorted(pairs)`.
- Mức 2: so `s <= out[-1][1]`.
- Mức 3: dùng `max` khi nối.

**Bài 4.**
- Mức 1: `next(it, None)` trả `None` khi hết.
- Mức 2: heap chứa `(s, e, i)` (đều là số) để so sánh an toàn.
- Mức 3: phần xử lý khe hở giữ nguyên.

**Bài 5.**
- Mức 1: tất cả bận ⇔ `busy == K` (vì mỗi người không có đoạn chồng nhau).
- Mức 2: duyệt các cặp thời điểm sự kiện liên tiếp `(a, b)`.
- Mức 3: gộp các khoảng liền kề trong kết quả.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert free_sort([[[1, 5]], [[6, 8], [10, 11]], [[2, 3], [12, 15]]]) == [[5, 6], [8, 10], [11, 12]]

# Bài 2
assert free_sort([[[0, 4], [7, 9]], [[3, 5]], [[10, 12]]]) == [[5, 7], [9, 10]]

# Bài 3
def merge_intervals_ans(pairs):
    out = []
    for s, e in sorted(pairs):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out
assert merge_intervals_ans([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals_ans([[1, 4], [4, 5]]) == [[1, 5]]

# Bài 4
def free_heap_iter_ans(raw):
    its = [iter(emp) for emp in raw]
    h = []
    for i, it in enumerate(its):
        first = next(it, None)
        if first:
            heapq.heappush(h, (first[0], first[1], i))
    res, prev = [], None
    while h:
        s, e, i = heapq.heappop(h)
        if prev is not None and s > prev:
            res.append([prev, s])
        prev = e if prev is None else max(prev, e)
        nxt = next(its[i], None)
        if nxt:
            heapq.heappush(h, (nxt[0], nxt[1], i))
    return res

# Bài 5
def common_busy_ans(raw):
    K = len(raw)
    delta = {}
    for emp in raw:
        for s, e in emp:
            delta[s] = delta.get(s, 0) + 1
            delta[e] = delta.get(e, 0) - 1
    ts = sorted(delta)
    res, busy = [], 0
    for a, b in zip(ts, ts[1:]):
        busy += delta[a]
        if busy == K:
            if res and res[-1][1] == a:
                res[-1][1] = b
            else:
                res.append([a, b])
    return res

def common_busy_brute(raw):
    pairs = [p for emp in raw for p in emp]
    lo, hi = min(s for s, _ in pairs), max(e for _, e in pairs)
    cnt = [0] * (hi - lo)
    for emp in raw:
        for s, e in emp:
            for t in range(s, e):
                cnt[t - lo] += 1
    res = []
    for t in range(hi - lo):
        if cnt[t] == len(raw):
            if res and res[-1][1] == t + lo:
                res[-1][1] += 1
            else:
                res.append([t + lo, t + lo + 1])
    return res

random.seed(99)
for _ in range(400):
    raw = gen_raw(random.randint(1, 4), random.randint(1, 4), 30, 4)
    assert free_heap_iter_ans(raw) == brute(raw)
    assert common_busy_ans(raw) == common_busy_brute(raw), raw
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (sắp + quét khe hở):**

```python
ivs = sorted((iv for emp in schedule for iv in emp), key=lambda iv: iv.start)
res, prev_end = [], ivs[0].end
for iv in ivs[1:]:
    if iv.start > prev_end:                    # khe hở thật (độ dài > 0)
        res.append(Interval(prev_end, iv.start))
    prev_end = max(prev_end, iv.end)           # luôn dùng max
return res
```

**Template heap (K danh sách đã sắp):** đẩy `(start, emp_id, idx)`; pop → xử lý → đẩy phần tử kế tiếp của cùng người.

**Checklist trước khi submit:**

- [ ] Làm việc với **đối tượng `Interval`** (`.start`, `.end`), trả về list `Interval`.
- [ ] **Không** khai báo lại lớp `Interval` khi nộp.
- [ ] So sánh `start > prev_end` (loại khoảng độ dài 0).
- [ ] `prev_end = max(prev_end, end)`.
- [ ] Không thêm khoảng vô hạn ở đầu/cuối.
- [ ] Heap: không để `Interval` bị so sánh trực tiếp.
- [ ] Tên method đúng: `employeeFreeTime(self, schedule)`.